In [1]:
import ast
import os
import pandas as pd
import textract

# chromid reference data

file_path = "/active-data/analysis_results/chr_pla/reference_data/PMID20080407/1-s2.0-S0966842X09002698-mmc1.doc"
content = textract.process(file_path, encoding="utf-8")
content = content.decode("utf-8")

ref1_list = []
for line in content.splitlines():
    if "Chromid" in line and '|' in line:
        parts = line.split("|")
        nc_code = parts[4].strip()
        ref1_list.append(nc_code.replace(' ', '_'))

file_path = "/active-data/analysis_results/chr_pla/reference_data/PMID40827884/msystems.00175-25-s0003.xlsx"

df = pd.read_excel(file_path, skiprows=1)
chromid_ref = df[df['GC_difference_%'] != 0]
ref2_list = chromid_ref['Replicon_Accession'].to_list()

# chromid-finder data

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
filted_data = all_data[all_data['genus_clean'].isin(keep_genus)]

found_chromids = []
chromid_dir = '/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/chromid_results'
for acc_n in filted_data['accession']:
    file_path = f'{chromid_dir}/{acc_n}.txt'
    
    if os.path.getsize(file_path) == 0:
        continue
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
        for i, line in enumerate(lines):
            if "Possible bacterial chromids:" in line:
                if i + 1 >= len(lines):
                    continue
                chromid_line = lines[i+1].strip()
                if not chromid_line or chromid_line == '------':
                    break
                
                chromid_list = [c.strip() for c in chromid_line.split(",") if c.strip()]
                for chromid in chromid_list:
                    found_chromids.append(chromid.split('.')[0])

In [2]:
all_data = []

for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    os.chdir(folder)
    replicon_data = pd.read_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv')
    replicon_data['genus'] = genus_name
    all_data.append(replicon_data)

all_data = pd.concat(all_data, ignore_index=True)
all_data['contig'] = all_data['accession'].apply(lambda x: x.split('-')[-1].split('.')[0])

In [3]:
map_dict = {
    "ref1_list": "PMID20080407",
    "ref2_list": "PMID40827884",
    "found_chromids": "chromid-finder"
}

def get_chromid_flag_and_ref(contig_id):
    tags = []
    if contig_id in ref1_list:
        tags.append(map_dict["ref1_list"])
    if contig_id in ref2_list:
        tags.append(map_dict["ref2_list"])
    if contig_id in found_chromids:
        tags.append(map_dict["found_chromids"])
    
    if len(tags) > 0:
        chromid_flag = "Yes"
        ref_str = ";".join(tags)
    else:
        chromid_flag = "No"
        ref_str = ""
    return chromid_flag, ref_str

all_data[["chromid", "ref_data"]] = all_data["contig"].apply(
    lambda x: pd.Series(get_chromid_flag_and_ref(x))
)

In [4]:
all_data = all_data[['accession', 'genus', 'size', 'category-pident_90', 'chromid', 'ref_data']]
all_data.rename(columns={'accession': 'contig'}, inplace=True)

target_dir = '/active-data/analysis_results/chr_pla/genus/suptables'
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)
all_data.to_csv('replicon_chromid_data.tsv', sep='\t', index=False)